In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os

# System definitions

In [ ]:
from nsflows.systems.gaussians import normal
from nsflows.systems.testsystems_2D import double_well

n_particles = 1
dimensions = 2

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

normal_2D = normal(n_particles=n_particles, dimensions=dimensions, device=device)
double_well_2D = double_well(n_particles=n_particles, dimensions=dimensions, device=device, eps=3., c=1., d=0.5)

# Plot PES

In [ ]:
x_min = -2.1
x_max = +2.1
y_min = -4.0
y_max = +3.5
n_grid = 200

x = np.linspace(x_min, x_max, n_grid)
y = np.linspace(y_min, y_max, n_grid)

X, Y = np.meshgrid(x, y)

Z_source = np.zeros([len(X),len(Y)])
Z_target = np.zeros([len(X),len(Y)])

for i in range(len(X)):
    for j in range(len(Y)):

        conf = torch.from_numpy(np.array([X[i][j], Y[i][j]], dtype=np.float32)).unsqueeze(0).to(device)

        Z_source[i,j] = normal_2D.energy(conf).squeeze().cpu().numpy()
        Z_target[i,j] = double_well_2D.energy(conf).squeeze().cpu().numpy()

In [ ]:
fig_size = (10 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400)


# ax.scatter(cpu_samples[:,0], cpu_samples[:,1], s=.25, zorder = 10)
ax.contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
# ax.contour(X, Y, Z_target, levels=[U_max], colors="C1", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xticks([])  # Remove x ticks
ax.set_yticks([])  # Remove y ticks

plt.show()

## Define Parameters

In [ ]:
# Load Previous Run
load = False

# Nested Sampling Parameters
live_samples = 1024

# The initial live-sample set is picked from data/dw/ according to live_samples.
init_samples_dir = "../data/dw"
init_samples_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", "samples_init.pt")
if not os.path.exists(init_samples_filepath):
    available = sorted(d[1:].lstrip("0") for d in os.listdir(init_samples_dir) if d.startswith("K"))
    raise FileNotFoundError(
        f"No initial samples shipped for live_samples={live_samples} "
        f"(looked for {init_samples_filepath}); available: {available}"
    )

# Number of nested sampling iterations, set per live-set size: a larger live set
# compresses the prior volume more slowly per iteration, so it needs more steps to
# reach the same depth. To run longer or shorter, edit the entry below or assign
# max_ns_iterations directly afterwards.
ns_iterations_for_K = {
    1024: 10000,     # 1e4 steps
    10000: 100000,   # 1e5 steps
}
if live_samples not in ns_iterations_for_K:
    raise KeyError(
        f"No iteration count defined for live_samples={live_samples}; "
        f"defined for {sorted(ns_iterations_for_K)}. Add an entry above, "
        "or set max_ns_iterations explicitly."
    )
max_ns_iterations = ns_iterations_for_K[live_samples]

n_propagate = 1000
update_step = True
turn_on_nf = -1

# Output Folder Definition

In [ ]:
from nsflows.tools.util import generate_unique_identifier, remove_empty_directories, generate_output_directory

remove_empty_directories("./output/")

if load:
    output_dir = "./output/..." # Insert output folder here if you want to load from an existing run. Otherwise, the code will generate a new run folder.
    print(f"Run Folder: {output_dir}")
else:
    run_id = generate_unique_identifier()
    output_dir = generate_output_directory(run_id)

In [ ]:
# ============================================================
# Generate summary for new runs
# OR
# Read and display existing summary for loaded runs
# ============================================================

from pathlib import Path
from datetime import datetime
import json

summary_txt_path = Path(output_dir) / "simulation_summary.txt"
summary_json_path = Path(output_dir) / "simulation_summary.json"

if not load:

    # --------------------------------------------------------
    # Build parameter summary dictionary
    # --------------------------------------------------------

    summary = {
        "timestamp": datetime.now().isoformat(),

        "system": {
            "name": "double_well",
            "n_particles": n_particles,
            "dimensions": dimensions,
            "eps": double_well_2D.eps,
            "c": double_well_2D.c,
            "d": double_well_2D.d,
            "device": str(device),
        },

        "nested_sampling": {
            "init_samples_filepath": init_samples_filepath,
            "live_samples": live_samples,
            "max_ns_iterations": max_ns_iterations,
            "n_propagate": n_propagate,
            "update_step": update_step,
            "turn_on_nf": turn_on_nf,
        },
    }

    # --------------------------------------------------------
    # Ensure output directory exists
    # --------------------------------------------------------

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Save JSON summary
    # --------------------------------------------------------

    with open(summary_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    # --------------------------------------------------------
    # Save readable text summary
    # --------------------------------------------------------

    with open(summary_txt_path, "w") as f:

        f.write("====================================================\n")
        f.write("Simulation Summary\n")
        f.write("====================================================\n\n")

        f.write(f"Generated: {summary['timestamp']}\n\n")

        f.write("SYSTEM PARAMETERS\n")
        f.write("-----------------\n")
        for k, v in summary["system"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nNESTED SAMPLING PARAMETERS\n")
        f.write("--------------------------\n")
        for k, v in summary["nested_sampling"].items():
            f.write(f"{k}: {v}\n")

    print(f"Summary files written to:\n")
    print(f"  JSON : {summary_json_path}")
    print(f"  TEXT : {summary_txt_path}")

else:

    # --------------------------------------------------------
    # Read and print existing summary
    # --------------------------------------------------------

    try:
        with open(summary_txt_path, "r") as f:
            summary_contents = f.read()

        print("====================================================")
        print("Loaded Run Summary")
        print("====================================================\n")

        print(summary_contents)

    except FileNotFoundError:
        print("WARNING: No simulation summary file found.")
        print(f"Expected location:\n{summary_txt_path}")

    except Exception as e:
        print("ERROR while reading simulation summary:")
        print(e)


# Nested Sampling

In [ ]:
from nsflows.samplers.monte_carlo import rejection_monte_carlo
from nsflows.nested_sampling import nested_sampling

rejection_sampler = rejection_monte_carlo(system=double_well_2D, n_cycles=100, step_size=1.2, transform=False)

samples, U_samples, acceptance, umax_plt = nested_sampling(K=live_samples, 
                                                        system=double_well_2D, 
                                                        std_propagator=rejection_sampler, 
                                                        init_samples_filepath=init_samples_filepath, 
                                                        max_iters=max_ns_iterations, 
                                                        n_propagate=n_propagate, 
                                                        update_step=update_step,
                                                        turn_on_nf=turn_on_nf, 
                                                        iprint=1, 
                                                        isavesamp=500,
                                                        save_biased_pool=True,
                                                        outputdir=output_dir)

## Live Sets

In [ ]:
# Live set, plotted roughly every `plot_every_iters` nested-sampling iterations.
# The run saves samples_<iter>.pt / U_max_<iter>.pt every `isavesamp` iterations, so
# the spacing is rounded up to whatever snapshots exist.
plot_every_iters = 1000

snapshots = sorted(f for f in os.listdir(output_dir)
                   if f.startswith("samples_") and f.endswith(".pt"))
if not snapshots:
    print(f"No sample snapshots found in {output_dir}")

last_plotted = None
for filename in snapshots:
    iteration = int(filename[len("samples_"):-len(".pt")])
    if last_plotted is not None and iteration - last_plotted < plot_every_iters:
        continue
    last_plotted = iteration

    live_set = torch.load(os.path.join(output_dir, filename))
    live_set = live_set.view(-1, double_well_2D.n_particles, double_well_2D.dimensions).cpu().numpy()

    # Energy bound at this iteration, drawn as the boundary contour.
    u_max_filepath = os.path.join(output_dir, f"U_max_{iteration:012d}.pt")
    u_max = torch.load(u_max_filepath).item() if os.path.exists(u_max_filepath) else None

    fig_size = (10 * 0.393701, 7 * 0.393701)
    fig, ax = plt.subplots(figsize = fig_size, dpi = 400)

    ax.scatter(live_set[:,:,0], live_set[:,:,1], s=.25, zorder = 10, alpha=1, color="C0")
    ax.contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
    if u_max is not None:
        ax.contour(X, Y, Z_target, levels=[u_max], colors="C4", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"Iteration {iteration}")

    plt.show()


## Energy vs. Iteration and Density of States vs. Iteration

In [ ]:
vals, bins = np.histogram(umax_plt-umax_plt.min(), bins=200)
bin_centers = .5*(bins[1:] + bins[:-1])

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# umax_plt covers the iterations that actually ran, which is fewer than
# max_ns_iterations if the run was stopped early.
iters = np.arange(len(umax_plt))/1e4

fig_size = (10 * 0.393701, 8 * 0.393701)
fig, ax = plt.subplots(1, 2, figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

ax[0].plot(iters, (umax_plt-umax_plt.min()), color="C0")

ax[0].set_ylabel("Energy")
ax[0].set_xlabel(r"Iteration ($\times 10^4$)")

ax_inset = inset_axes(ax[0], width="50%", height="30%", loc='upper right', borderpad=.5)
ax_inset.plot(iters, (umax_plt-umax_plt.min()), color="C0")
ax_inset.set_yscale("log")
ax_inset.set_ylim(.9e-2,1.6e2)

ax[1].plot(vals, bin_centers, color="C4")
ax[1].set_xlabel(r"Samples")
ax[1].set_xscale('log')

plt.show()


## Timings

In [ ]:
# Elapsed time for this run, read from the timings file written by nested_sampling.
timings_filepath = os.path.join(output_dir, "timings.txt")

with open(timings_filepath) as f:
    notes = [line.lstrip("#").strip() for line in f if line.startswith("#")]

for key in ("Run started", "Run ended", "Elapsed time", "Run interrupted"):
    for note in notes:
        if note.startswith(key):
            print(note)

# Without the flow, STD_NS is the only non-zero column: the time spent sampling.
sampling_time = np.genfromtxt(timings_filepath, skip_header=2, names=True, comments="#")["STD_NS"].sum()
n_iters = len(umax_plt) - 1

print(f"Iterations completed: {n_iters}")
if n_iters > 0:
    print(f"Sampling time: {sampling_time:.1f} s ({1e3*sampling_time/n_iters:.2f} ms per iteration)")
